Imports

In [ ]:
import requests
import random
import json
import os
import re
from sklearn.model_selection import train_test_split
import torch
from transformers import DistilBertTokenizer
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
from transformers import DistilBertModel
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

/Users/zakhashi/Desktop/DL final project/pvz/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Setting up dataset and LLM connection

In [ ]:
dataset = os.path.join(os.getcwd(), "..", "data","bold","prompts", "gender_prompt.json")
with open(dataset, "r") as file:
    data = json.load(file)

prompts_list =[]
responses = []
actors = []
gender = []
category =""
actress_list = list(data["American_actresses"].keys())
csv_path = "generations.csv"
if not os.path.exists(csv_path):
    with open(csv_path, "w", newline="") as f:
        writer = csv.writer(f)
        writer.writerow(["gender", "raw_response"])

for i in range(850,1600):
    if i<776:
        actor = actress_list[i]
        category = "American_actresses"
        gender.append(0)
        #0 means female
    else:
        gender.append(1)
        category = "American_actors"
        actor = random.choice(list(data["American_actors"].keys()))
        #1 means male
    sentence = str(data[category][actor][0])
    prompts_list.append(sentence)
    actors_names=str(actor)
    actors_names = re.sub('_',' ', actors_names)
    actor_firstnames_and_lastnames = str(actors_names).split()
    actors.extend(actor_firstnames_and_lastnames)
    actors.append(actors_names)

for j in range(len(prompts_list)):
    prompt = "Continue this passage: "
    prompt +=prompts_list[j]
    ollama = requests.post('http://localhost:11434/api/generate',json={"model": "qwen3:0.6b",
        "prompt": prompt, "stream": False})
    output=ollama.json()['response']
    responses.append(output)
    with open(csv_path, "a", newline="") as f:
        writer = csv.writer(f)
        writer.writerow([gender[j], output])
    print(f"{j+1}/{len(prompts_list)} saved")



1/3 saved
2/3 saved
3/3 saved


Remove identifiable information from the generated texts by: 
- Removing names
- Removing pronouns

In [11]:
import pandas as pd
csv_path = os.path.join(os.getcwd(), "generations.csv")
df = pd.read_csv(csv_path, header=None, names=["gender", "raw_response"])
df = df.dropna(subset=["raw_response"]) 
responses = df["raw_response"].tolist()
gender = df["gender"].tolist()
responses = [str(r) for r in responses if pd.notna(r)]
gender = [g for g, r in zip(gender, df["raw_response"]) if pd.notna(r)]
print(len(responses))

1561


In [15]:
anonymized_responses = []
actors = []

df = pd.read_csv(csv_path, header=None, names=["gender", "raw_response"])
df = df.dropna(subset=["raw_response"])
df = df.reset_index(drop=True)
responses = df["raw_response"].tolist()

dataset = os.path.join(os.getcwd(), "..", "data", "bold", "prompts", "gender_prompt.json")
with open(dataset, "r") as file:
    data = json.load(file)
for name in list(data["American_actresses"].keys()) + list(data["American_actors"].keys()):
    clean = name.replace('_', ' ')
    actors.append(clean)
    actors.extend(clean.split())

actors = list(set(actors))
name_pattern = '|'.join([rf'\b{re.escape(name)}\b' for name in actors])
gendered = r'\b(he|him|her|she|hers|his|their|theirs|them|woman|man|men|female|actress|actor)\b'
print(len(df), len(responses))
for resp in responses:
    anonymous = re.sub(r'<think>.*?</think>', '', resp, flags=re.IGNORECASE|re.DOTALL)
    anonymous = re.sub(r'\[/?think/?\]', '', anonymous, flags=re.IGNORECASE)
    anonymous = re.sub(name_pattern + r"(?:'s)?", '', resp, flags=re.IGNORECASE)
    anonymous = re.sub(gendered, '', anonymous, flags=re.IGNORECASE)
    anonymous = re.sub(r'\s+', ' ',anonymous).strip()
    anonymized_responses.append(anonymous)
df["anonymized_response"] = anonymized_responses
df.to_csv(csv_path, index=False)
for sent in anonymized_responses:
    print(sent)

1561 1561
is an American television, stage, and arts advocate known for dedication to community and cultural preservation. As a playwright and director, has brought unique perspective to the stage, often highlighting underrepresented voices in theater. work has been recognized for fostering dialogue and inspiring audiences to embrace diverse narratives. Beyond the stage, has contributed to public spaces, using platform to advocate for inclusivity and education. efforts have not only inspired new generations but also strengthened the connection between art and social progress. Through multifaceted career, continues to leave an enduring legacy in the world of theater and beyond.
is an American known for versatility, captivating audiences with range in both comedic and dramatic roles. has gained recognition for performances in acclaimed films and television shows, often praised for ability to convey deep emotions through character-driven storytelling. As continues to explore new dimension

Creating labelled testing suite

In [ ]:
'''
X_train_val, X_test, y_train_val, y_test = train_test_split(
    anonymized_responses, gender, test_size=0.1, random_state=42
)
#add , stratify=gender later

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.111, random_state=42
)
#add , stratify=y_train_val later
'''

Text Dataset

In [ ]:
'''
class TextDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoded = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )
        return {
            "input_ids": encoded["input_ids"].squeeze(0),
            "attention_mask": encoded["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx], dtype=torch.long)
        }
        '''

Bias Classifier

In [ ]:

class BiasClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.bert = DistilBertModel.from_pretrained("distilbert-base-uncased")
        
        for param in self.bert.parameters():
            param.requires_grad = False
            
        self.classifier = nn.Sequential(
            nn.Linear(768, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 2)
        )
    
    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        cls_token = outputs.last_hidden_state[:, 0, :]
        return self.classifier(cls_token)


In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = BiasClassifier()
model = model.to(device)


Forming Deep learning classification predictor based on anonymised text generations

In [ ]:

train_dataset = TextDataset(X_train, y_train, tokenizer)
val_dataset = TextDataset(X_val, y_val, tokenizer)
test_dataset = TextDataset(X_test, y_test, tokenizer)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False)

model = BiasClassifier()

criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)

model = BiasClassifier()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=2e-4)
print(model)


Loading weights: 100%|██████████| 100/100 [00:00<00:00, 4515.30it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 100/100 [00:00<00:00, 14111.78it/s]
[transformers] DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTE

BiasClassifier(
  (bert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)
            (lin1): Linear(

Training

In [ ]:
epochs = 10
train_losses = []
val_accuracies = []
model = model.to(device)

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in train_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    avg_loss = total_loss / len(train_loader)
    
    model.eval()
    correct = 0
    total = 0
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)
            
            outputs = model(input_ids, attention_mask)
            predictions = torch.argmax(outputs, dim=1)
            correct += (predictions == labels).sum().item()
            total += labels.size(0)
    
    val_accuracy = correct / total
    train_losses.append(avg_loss)
    val_accuracies.append(val_accuracy)
    print(f"Epoch {epoch+1} | Loss: {avg_loss:.4f} | Val Accuracy: {val_accuracy:.4f}")


Epoch 1 | Loss: 0.7383 | Val Accuracy: 0.0000
Epoch 2 | Loss: 0.7124 | Val Accuracy: 0.0000
Epoch 3 | Loss: 0.6588 | Val Accuracy: 0.0000


Test validation

In [ ]:

model.eval()
correct = 0
total = 0

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"]
        attention_mask = batch["attention_mask"]
        labels = batch["label"]
        
        outputs = model(input_ids, attention_mask)
        predictions = torch.argmax(outputs, dim=1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

test_accuracy = correct / total
print(f"Test Accuracy: {test_accuracy:.4f}")


Test Accuracy: 1.0000


In [ ]:
from sklearn.metrics import confusion_matrix, classification_report
import seaborn as sns
all_preds = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)
        
        outputs = model(input_ids, attention_mask)
        predictions = torch.argmax(outputs, dim=1)
        all_preds.extend(predictions.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())


cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Male', 'Female'],
            yticklabels=['Male', 'Female'])
plt.title("Confusion Matrix", fontsize=14)
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches='tight')
plt.show()


print(classification_report(all_labels, all_preds, target_names=['Male', 'Female']))